#### **Scrapping Real estate Data from property24 for https://www.property24.co.ke/**

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

**single page scrapping**

In [ ]:

url="https://www.property24.co.ke/property-for-sale-in-nairobi-c1890"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")
properties = soup.find_all("div", class_="p24_regularTile")
data = []

for property in properties:

    title = property.find("span", class_="p24_propertyTitle")

    price = property.find("span", class_="p24_price")

    location = property.find("span", class_="p24_location")

    data.append({
        "title": title.text.strip() if title else None,
        "price": price.text.strip() if price else None,
        "location": location.text.strip() if location else None
    })

200


In [3]:
df = pd.DataFrame(data)

print(df.head())

                        title           price    location
0  2 Bedroom Apartment / Flat   KSh 8 850 000   Westlands
1  2 Bedroom Apartment / Flat  KSh 11 200 000  Kileleshwa
2  2 Bedroom Apartment / Flat  KSh 11 390 000  Kileleshwa
3  2 Bedroom Apartment / Flat  KSh 12 420 000  Kileleshwa
4  4 Bedroom Apartment / Flat  KSh 55 000 000   Westlands


**multipage for single category scrapping**

In [4]:
all_data = []

for page in range(1, 20):

    url = f"https://www.property24.co.ke/property-for-sale-in-nairobi-c1890?Page=1"

    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, "html.parser")

    properties = soup.find_all("div", class_="p24_regularTile")

    for property in properties:

        title = property.find("span", class_="p24_propertyTitle")

        price = property.find("span", class_="p24_price")

        location = property.find("span", class_="p24_location")

        all_data.append({
            "title": title.text.strip() if title else None,
            "price": price.text.strip() if price else None,
            "location": location.text.strip() if location else None
        })

df = pd.DataFrame(all_data)
df

#df.to_csv("nairobi_properties.csv", index=False)

,title,price,location
0,2 Bedroom Apartment / Flat,KSh 8 850 000,Westlands
1,2 Bedroom Apartment / Flat,KSh 11 200 000,Kileleshwa
2,2 Bedroom Apartment / Flat,KSh 11 390 000,Kileleshwa
3,2 Bedroom Apartment / Flat,KSh 12 420 000,Kileleshwa
4,4 Bedroom Apartment / Flat,KSh 55 000 000,Westlands
...,...,...,...
394,3 Bedroom Apartment / Flat,KSh 39 500 000,Westlands
395,3 Bedroom Apartment / Flat,KSh 51 200 000,Westlands
396,4 Bedroom Apartment / Flat,KSh 65 800 000,Westlands
397,1 Bedroom Apartment / Flat,KSh 5 000 000,Westlands


In [5]:
df['location'].unique()

<ArrowStringArray>
['Westlands', 'Kileleshwa']
Length: 2, dtype: str

**MULTIPAGE FOR DIFFERENT CATEGORIES SCRAPPING**

defining categories

In [6]:
#defining categories
categories = {
    "apartments": "https://www.property24.co.ke/apartments-flats-for-sale-in-nairobi-c1890",
    "houses": "https://www.property24.co.ke/houses-for-sale-in-nairobi-c1890",
    "townhouses": "https://www.property24.co.ke/townhouses-for-sale-in-nairobi-c1890",
    "plots": "https://www.property24.co.ke/townhouses-for-sale-in-nairobi-c1890",
    "farms": "https://www.property24.co.ke/farms-for-sale-in-nairobi-c1890",
    "commercial_property":"https://www.property24.co.ke/commercial-property-for-sale-in-nairobi-c1890",
    "industrial_property":"https://www.property24.co.ke/industrial-property-for-sale-in-nairobi-c1890",
}

creating a scraping function

In [7]:

headers = {"User-Agent": "Mozilla/5.0"}

def scrape_category(url, category_name, pages=3):
    data = []

    for page in range(1, pages + 1):

        paginated_url = f"{url}?Page={page}"

        response = requests.get(paginated_url, headers=headers)
        soup = BeautifulSoup(response.text, "lxml")

        listings = soup.find_all("div", class_="p24_regularTile")

        for item in listings:

            title = item.find("span", class_="p24_propertyTitle")
            price = item.find("span", class_="p24_price")
            location = item.find("span", class_="p24_location")

            data.append({
                "category": category_name,
                "title": title.text.strip() if title else None,
                "price": price.text.strip() if price else None,
                "location": location.text.strip() if location else None
            })

        time.sleep(2)  # avoid blocking

    return data

looping through all categories

In [8]:
all_data = []

for category, url in categories.items():
    print(f"Scraping {category}...")
    category_data = scrape_category(url, category, pages=5)
    all_data.extend(category_data)

df = pd.DataFrame(all_data)

df

Scraping apartments...
Scraping houses...
Scraping townhouses...
Scraping plots...
Scraping farms...
Scraping commercial_property...
Scraping industrial_property...


,category,title,price,location
0,apartments,2 Bedroom Apartment / Flat,KSh 8 850 000,Westlands
1,apartments,2 Bedroom Apartment / Flat,KSh 11 200 000,Kileleshwa
2,apartments,2 Bedroom Apartment / Flat,KSh 11 390 000,Kileleshwa
3,apartments,2 Bedroom Apartment / Flat,KSh 12 420 000,Kileleshwa
4,apartments,4 Bedroom Apartment / Flat,KSh 55 000 000,Westlands
...,...,...,...,...
675,industrial_property,Industrial Property,KSh 110 000 000,Industrial Area
676,industrial_property,Industrial Property,KSh 45 000 000,Runda
677,industrial_property,Industrial Property,KSh 95 000,Lavington
678,industrial_property,Industrial Property,KSh 25 000 000,Syokimau


In [9]:
df.to_csv("property24_all_categories.csv", index=False)